In [ ]:
# ── Imports and paths ──

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn2
from scipy.stats import hypergeom
from scipy.stats import mannwhitneyu
import os

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
OUT = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

SHAP_RANKED  = os.path.join(OUT, "delta_feature_importances.csv")
LME_SIG      = os.path.join(OUT, "lme_significant_otus_annotated.csv")
LME_ALL      = os.path.join(OUT, "lme_interaction_results.csv")

OUT_TABLE    = os.path.join(OUT, "lme_shap_overlap_table.csv")
OUT_FIGURE   = os.path.join(OUT, "lme_shap_overlap.png")

print("Paths set.")
print(f"  Base : {BASE_DIR}")
print(f"  Out  : {OUT}")

In [ ]:
# ── Load files ──

# SHAP rankings from delta RF (9,612 OTUs)
shap_df = pd.read_csv(SHAP_RANKED)
print(f"SHAP rankings    : {shap_df.shape}  cols: {list(shap_df.columns)}")

# LME significant OTUs (96, FDR < 0.05, with taxonomy)
lme_sig = pd.read_csv(LME_SIG)
print(f"LME significant  : {lme_sig.shape}  cols: {list(lme_sig.columns)}")

# Full LME tested OTUs (200, background set)
lme_all = pd.read_csv(LME_ALL)
print(f"LME all OTUs     : {lme_all.shape}  cols: {list(lme_all.columns)}")

In [ ]:
# ── Define OTU sets and background ──

# Background = 200 OTUs tested in LME
background    = set(lme_all['OTU_ID'])
N_background  = len(background)

# LME significant OTUs within background
lme_sig_ids   = set(lme_sig['OTU_ID']) & background
K_lme_sig     = len(lme_sig_ids)

# SHAP ranks — restrict to background only
shap_bg = (
    shap_df[shap_df['OTU_ID'].isin(background)]
    .sort_values('rank')
    .reset_index(drop=True)
)

print(f"Background OTUs (LME tested)       : {N_background}")
print(f"LME significant within background  : {K_lme_sig}")
print(f"SHAP OTUs within background        : {len(shap_bg)}")
print(f"\nSHAP rank range within background  : {shap_bg['rank'].min()} — {shap_bg['rank'].max()}")

In [ ]:
# ── Hypergeometric overlap test ──

results = []

for cutoff in [50, 100]:
    top_shap    = set(shap_bg.nsmallest(cutoff, 'rank')['OTU_ID'])
    overlap     = top_shap & lme_sig_ids
    expected    = round(K_lme_sig * cutoff / N_background, 2)

    # Hypergeometric test
    # P(X >= observed | N_background, K_lme_sig, cutoff)
    p_val = hypergeom.sf(len(overlap) - 1, N_background, K_lme_sig, cutoff)

    results.append({
        'shap_cutoff'  : cutoff,
        'n_top_shap'   : cutoff,
        'n_lme_sig'    : K_lme_sig,
        'n_overlap'    : len(overlap),
        'expected'     : expected,
        'p_hypergeom'  : p_val,
        'overlap_ids'  : overlap,
    })

    print(f"Top-{cutoff:>3} SHAP  |  "
          f"overlap = {len(overlap):>2}  |  "
          f"expected = {expected}  |  "
          f"hypergeometric p = {p_val:.4e}")

In [ ]:
# ── Mann–Whitney U rank test ──

# Tag each OTU in full SHAP list as LME significant or not
shap_df['lme_significant'] = shap_df['OTU_ID'].isin(lme_sig_ids)

sig_ranks   = shap_df[shap_df['lme_significant'] == True]['rank'].values
nonsig_ranks = shap_df[shap_df['lme_significant'] == False]['rank'].values

print(f"LME significant OTUs   : {len(sig_ranks)}")
print(f"LME non-significant    : {len(nonsig_ranks)}")
print(f"\nMedian SHAP rank — LME significant : {np.median(sig_ranks):.1f}")
print(f"Median SHAP rank — LME non-sig     : {np.median(nonsig_ranks):.1f}")
print(f"(Lower rank = higher SHAP importance)")

# Mann-Whitney U test
U, p = mannwhitneyu(sig_ranks, nonsig_ranks, alternative='less')
print(f"\nMann-Whitney U test (one-sided: sig ranks lower than non-sig)")
print(f"  U statistic : {U:.2f}")
print(f"  p-value     : {p:.4e}")

In [ ]:
# ── Annotated overlap table ──

def parse_genus(tax_str):
    if pd.isna(tax_str):
        return 'Unknown'
    for part in tax_str.split(';'):
        part = part.strip()
        if part.startswith('g__'):
            return part[3:].strip() or 'Unknown'
    return 'Unknown'

def parse_species(tax_str):
    if pd.isna(tax_str):
        return 'Unknown'
    for part in tax_str.split(';'):
        part = part.strip()
        if part.startswith('s__'):
            return part[3:].strip() or 'Unknown'
    return 'Unknown'

# Merge LME significant with SHAP ranks
overlap_df = (
    lme_sig[['OTU_ID', 'estimate', 'q_value', 'taxonomy']]
    .merge(
        shap_df[['OTU_ID', 'rank', 'mean_abs_shap']],
        on='OTU_ID', how='left'
    )
    .sort_values('rank')
    .reset_index(drop=True)
)

overlap_df['genus']   = overlap_df['taxonomy'].apply(parse_genus)
overlap_df['species'] = overlap_df['taxonomy'].apply(parse_species)

# Save
overlap_df.to_csv(OUT_TABLE, index=False)
print(f"Saved: {OUT_TABLE}")
print(f"\nTop 20 convergent OTUs by SHAP rank:")
print(overlap_df[['rank', 'genus', 'species', 'estimate', 'q_value']]
      .head(20).to_string(index=False))

In [ ]:
# ── Save Mann–Whitney result and plot ──

# ── Save statistical result ───────────────────────────────
stats_out = os.path.join(OUT, "lme_shap_mwu_result.txt")
with open(stats_out, 'w') as f:
    f.write("LME × SHAP Convergence — Mann-Whitney U Test\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Test: Do LME significant OTUs rank higher in SHAP importance?\n")
    f.write(f"Alternative hypothesis: one-sided (sig ranks lower than non-sig)\n\n")
    f.write(f"LME significant OTUs          : {len(sig_ranks)}\n")
    f.write(f"LME non-significant OTUs      : {len(nonsig_ranks)}\n")
    f.write(f"Total SHAP-ranked OTUs        : {len(shap_df)}\n\n")
    f.write(f"Median SHAP rank (LME sig)    : {np.median(sig_ranks):.1f}\n")
    f.write(f"Median SHAP rank (LME non-sig): {np.median(nonsig_ranks):.1f}\n\n")
    f.write(f"U statistic                   : {U:.2f}\n")
    f.write(f"p-value                       : {p:.4e}\n")

print(f"Saved: {stats_out}")

# ── Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('LME × SHAP Convergence', fontsize=14, fontweight='bold')

# ── Panel A: Venn diagram ─────────────────────────────────
ax = axes[0]
ax.axis('off')

top100      = set(shap_bg.nsmallest(100, 'rank')['OTU_ID'])
only_shap   = len(top100 - lme_sig_ids)
only_lme    = len(lme_sig_ids - top100)
both        = len(top100 & lme_sig_ids)

v = venn2(
    subsets=(only_shap, only_lme, both),
    set_labels=('Top-100\nSHAP', 'LME\nFDR<0.05'),
    ax=ax,
    set_colors=('#5B8DB8', '#E07B54'),
    alpha=0.6
)
ax.set_title(
    f'A   Top-100 SHAP ∩ LME significant\n'
    f'(Mann-Whitney p = {p:.2e})',
    fontsize=11, loc='left'
)

# ── Panel B: Top 20 convergent OTUs by LME effect size ───
ax = axes[1]
plot_df     = overlap_df.head(20).sort_values('estimate', ascending=True)
bar_colors  = ['#E07B54' if e > 0 else '#5B8DB8' for e in plot_df['estimate']]

y_pos = range(len(plot_df))
ax.barh(list(y_pos), plot_df['estimate'].values,
        color=bar_colors, edgecolor='none', height=0.7)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(
    [f"{g} (SHAP rank {r})" for g, r in
     zip(plot_df['genus'], plot_df['rank'])],
    fontsize=8
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('LME interaction estimate (CLR)', fontsize=10)
ax.set_title('B   Top-20 convergent OTUs by SHAP rank', fontsize=11, loc='left')
ax.spines[['top', 'right']].set_visible(False)

pos_patch = mpatches.Patch(color='#E07B54', label='Fiber-enriched')
neg_patch = mpatches.Patch(color='#5B8DB8', label='Fiber-depleted')
ax.legend(handles=[pos_patch, neg_patch], frameon=False, fontsize=9,
          bbox_to_anchor=(0.5, -0.08), loc='upper center', ncol=2)

plt.tight_layout()
plt.savefig(OUT_FIGURE, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_FIGURE}")

In [ ]:
# ── Summary ──

print("=" * 60)
print("  NOTEBOOK 08 — LME × SHAP CONVERGENCE SUMMARY")
print("=" * 60)

print(f"""
── Input ───────────────────────────────────────────────────
  SHAP-ranked OTUs (delta RF)     : {len(shap_df)}
  LME tested OTUs (background)    : {N_background}
  LME significant OTUs (FDR<0.05) : {K_lme_sig}

── Convergence Test ────────────────────────────────────────
  Median SHAP rank — LME sig      : {np.median(sig_ranks):.1f}
  Median SHAP rank — LME non-sig  : {np.median(nonsig_ranks):.1f}
  Mann-Whitney U                  : {U:.2f}
  p-value (one-sided)             : {p:.4e}

── Overlap at Top-100 SHAP ─────────────────────────────────
  Overlapping OTUs                : 49 / 96 LME significant
  Top convergent taxa             : Bifidobacterium,
                                    Eubacterium_F,
                                    Faecalibacterium prausnitzii,
                                    Roseburia hominis,
                                    Mediterraneibacter torques (depleted)

── Output Files ────────────────────────────────────────────
  {os.path.basename(OUT_TABLE)}
  {os.path.basename(OUT_FIGURE)}
  lme_shap_mwu_result.txt
""")

print("=" * 60)
print("  Notebook complete.")
print("=" * 60)